# Applied Workflows: Algorithms, File Persistence, and Visualization

[Back to Session Overview](../README.md) · [Back to Module 01](../../README.md)

This capstone notebook brings together sequence data structures, algorithmic paradigms, disk persistence, and visual analytics into an applied workflow.

### Learning Objectives
1. Implement and analyze sequence algorithms: Linear Search, Selection Sort, and Prefix Sum Arrays.
2. Leverage the Prefix Sum array (`accumulate`) to achieve $O(1)$ range sum query execution.
3. Manage file input/output safely using context managers (`with open(...)`).
4. Load and validate external tabular and categorical datasets from `data/`.
5. Plot continuous mathematical curves and categorical distributions using `matplotlib.pyplot` and `collections.Counter`.

In [ ]:
# Sequence Algorithms: Linear Search, Selection Sort, and Prefix Sum Array


def linear_search(arr: list[int], target: int) -> int:
    for idx, value in enumerate(arr):
        if value == target:
            return idx
    return -1


def selection_sort_naive(arr: list[int]) -> list[int]:
    # Naive sorting using min(), remove(), and append(): O(n^2)
    pool = arr.copy()
    sorted_result = []
    while pool:
        m = min(pool)
        pool.remove(m)
        sorted_result.append(m)
    return sorted_result


def compute_prefix_sums(arr: list[float]) -> list[float]:
    if not arr:
        return []
    prefix = [arr[0]]
    for i in range(1, len(arr)):
        prefix.append(prefix[i - 1] + arr[i])
    return prefix


def range_sum_query(prefix_sums: list[float], left: int, right: int) -> float:
    if left == 0:
        return prefix_sums[right]
    return prefix_sums[right] - prefix_sums[left - 1]


# Verify algorithms
test_seq = [6, 5, 7, 1, 9, 2]
assert linear_search(test_seq, 9) == 4
assert linear_search(test_seq, 99) == -1
assert selection_sort_naive(test_seq) == [1, 2, 5, 6, 7, 9]

# Prefix sum verification: sum from index 1 to 3: 5 + 7 + 1 = 13
pref = compute_prefix_sums([float(x) for x in test_seq])
assert abs(range_sum_query(pref, 1, 3) - 13.0) < 1e-9
print("Algorithm assertions passed successfully.")

## File Handling and Persistence with Context Managers

The Python `with open(filepath, mode, encoding='utf-8') as f:` statement enforces deterministic resource release via context management protocols (`__enter__` and `__exit__`), guaranteeing that file descriptors are immediately closed even if runtime exceptions occur.

In [ ]:
from pathlib import Path

DATA_DIR = Path("../data")
if not DATA_DIR.exists():
    DATA_DIR = Path("python-to-ai/01-python-fundamentals/2026-10-25-data-structure/data")

sales_file = DATA_DIR / "sales_records.txt"
feedback_file = DATA_DIR / "feedback.txt"

# Read and parse sales numbers
sales_values: list[float] = []
with sales_file.open("r", encoding="utf-8") as f:
    for line in f:
        cleaned = line.strip()
        if cleaned:
            sales_values.append(float(cleaned))

print(f"Loaded {len(sales_values)} sales records: {sales_values}")
assert len(sales_values) == 10
assert sales_values[0] == 120.50

# Read and parse categorical feedback
feedback_items: list[str] = []
with feedback_file.open("r", encoding="utf-8") as f:
    for line in f:
        item = line.strip()
        if item:
            feedback_items.append(item)

print(f"Loaded {len(feedback_items)} feedback responses.")
assert len(feedback_items) == 25
print("Dataset ingestion validated successfully.")

## Data Visualization with Matplotlib

Visualizing analytical functions and empirical distributions using `matplotlib.pyplot`.
- Plotting $y = x^2$ over the continuous domain $x \in [-8, 8]$.
- Visualizing categorical feedback frequencies as a bar histogram.

In [ ]:
from collections import Counter

import matplotlib.pyplot as plt


# 1. Plotting quadratic curve y = x^2
def linear_space(start: float, stop: float, num: int) -> list[float]:
    step = (stop - start) / (num - 1)
    return [start + i * step for i in range(num)]


x_coords = linear_space(-8.0, 8.0, 100)
y_coords = [x * x for x in x_coords]

plt.figure(figsize=(7, 3.5))
plt.plot(x_coords, y_coords, label="$y = x^2$", color="#1f77b4", linewidth=2)
plt.title("Quadratic Function over Domain $[-8, 8]$")
plt.xlabel("x values")
plt.ylabel("y values")
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()

# 2. Bar chart of feedback frequencies
counts = Counter(feedback_items)
categories = list(counts.keys())
frequencies = list(counts.values())

plt.figure(figsize=(8, 4))
plt.bar(categories, frequencies, color="#2ca02c", edgecolor="black", alpha=0.8)
plt.title("Customer Feedback Category Distribution")
plt.xlabel("Category")
plt.ylabel("Occurrence Count")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

assert counts["product quality"] == 11
print("Plotting code executed and assertions verified.")

## Practical Lab Exercise: Analytical Metric Pipeline

Implement `analyze_dataset_and_range_metrics(sales: list[float], feedback: list[str]) -> dict` that outputs comprehensive statistics and supports constant-time range sum querying.

In [ ]:
def analyze_dataset_and_range_metrics(sales: list[float], feedback: list[str]) -> dict:
    pref = compute_prefix_sums(sales)
    feedback_dist = Counter(feedback)
    return {
        "total_sales": sum(sales),
        "mean_sales": sum(sales) / len(sales) if sales else 0.0,
        "min_sales": min(sales) if sales else 0.0,
        "max_sales": max(sales) if sales else 0.0,
        "prefix_sums": pref,
        "feedback_counts": dict(feedback_dist),
        "dominant_feedback": feedback_dist.most_common(1)[0][0] if feedback_dist else None,
    }


report = analyze_dataset_and_range_metrics(sales_values, feedback_items)
assert abs(report["total_sales"] - 3237.8) < 1e-4
assert report["dominant_feedback"] == "product quality"
assert len(report["prefix_sums"]) == 10
print("Capstone analytics pipeline verified successfully!")